# Example 1: quickstart — load, inspect, then choose whether to fit

Welcome to Spyctres!

This example uses a clean bundled Gaia FGK Benchmark Stars spectrum, **18 Sco / HIP79672**, so the first plot should look sane before we introduce more complicated spectra, archive masks, etc.

## What this example teaches

- how to load a supported spectrum with `reader=`;
- how to inspect coverage, metadata, and diagnostic windows;
- how to build a first `FitSetup` without needing to run PHOENIX.

## Requirements

The default path uses bundled data only. The optional fit section requires a configured local PHOENIX library.

## Expected outputs

One spectrum plot, one diagnostic-window plot, and a compact setup summary. If `RUN_FIT=True`, the notebook also prints a first-pass model-fit summary and plot.

## Don't jump to conclusions

A visually reasonable first-pass fit is not a validation of Spyctres or a final atmospheric-parameter analysis.

The mental model is deliberately small:

```python
import Spyctres as sp

# Read the spectrum
spec = sp.read_spectrum("my_spectrum.fits", reader="xshooter_merge1d")
# Lets Spyctres inspect the spectrum and make suggestions for the fit
setup = sp.suggest_fit_setup(spec, intent="quicklook_classification")
# Perform the fit using the PHOENIX templates
result = sp.fit_stellar_spectrum(spec, model="phoenix", setup=setup)
# Plot the result for visual inspection
sp.plot_fit_referee(result)
```

Note that this notebook will not execute the expensive PHOENIX step, unless you explicitly set `RUN_FIT = True`.


## 1. Import Spyctres and choose the bundled spectrum

A `reader` is the profile that tells Spyctres how to interpret a particular **data product**: columns, FITS conventions, wavelength units, default frame metadata, and any known product-level caveats. It is more specific than just the telescope or instrument name.

You can discover available readers with `sp.list_readers()` and inspect one with `sp.get_reader_info("reader_name")`. If your spectrum is not supported yet, the safest path is to convert it to wavelength/flux/error/mask arrays and open an issue or add a small reader with explicit metadata.


In [ ]:
import Spyctres as sp

# Path to the spectrum file (we use the example spectra here)
spectrum_path = sp.example_data_path(
    "gaia_benchmark/HIP79672_HARPS_1_R42KNorm.txt.gz"
)

# Specify the appropriate reader to use for this spectrum
reader = "gbs_v3_ascii"

print("Available readers:")
print(", ".join(sp.list_readers()))
print()
# To get more information about a particular reader, you can use
print("Extended info of Reader used here:")
sp.get_reader_info(reader)

## 2. Read and inspect the spectrum

`read_spectrum()` loads the data into Spyctres’ standard spectrum format. In Spyctres, `valid_mask=True` means that a pixel is available for analysis. `spec.summary()` provides a concise overview, while `spec.provenance_summary()` shows more detailed information about the file and how Spyctres interpreted it.


In [ ]:
# Read the spectrum with the reader we specified
spec = sp.read_spectrum(spectrum_path, reader=reader)

# Print some info about the spectrum
# Short summary
print(spec.summary())
print()
# uncomment the line below for the longer summary -->
# print(spec.provenance_summary())

# Plot the full spectrum for a quick view
sp.plot_spectrum(
    spec,
    title="Example 1: 18 Sco / HIP79672 benchmark spectrum",
)

## 3. Ask Spyctres which regions are worth inspecting

Next we can ask Spyctres to highlight a few useful spectral regions that are covered by the data. These regions come from Spyctres’ built-in catalogue of common diagnostic features.

The orange bands mark places worth inspecting. They do not determine the stellar type automatically, and highlighting them does not mask, rescale, or otherwise change the spectrum.


In [ ]:
# Ask Spyctres to suggest up to six useful spectral regions
# that are covered by this spectrum
windows = sp.select_diagnostic_windows(
    spec,
    max_windows=6,
)

# Print a short summary of the suggested regions
print(windows.summary_text(max_rows=6))

# Plot the spectrum and highlight the suggested regions
# The orange bands are for visual inspection only:
# they do not mask, rescale, or modify the spectrum
sp.plot_diagnostic_windows(
    spec,
    selection=windows,
    title="Example 1: suggested diagnostic windows",
)

We can also zoom in on one of the suggested regions. Let's display the highest-ranked window so that we can inspect it more closely.

In [ ]:
# Select the first two highest-ranked diagnostic windows suggested by Spyctres
selected_windows = windows["selected"][:2]

# This example contains one spectrum segment
segment = spec.segments[0] if hasattr(spec, "segments") else spec

# Plot only the selected region so we can inspect the feature more closely
sp.plot_spectrum_line_windows(
    segment.wave,
    segment.flux,
    selected_windows,
    valid_mask=segment.valid_mask,
    title=f"Example 1: closer look at the selected windows",
    ncols=2 # set this to the number of subplots you want to plot (here we use 2)
)

You can also specify the wavelength range you want yourself

In [ ]:
sp.plot_spectrum_line_windows(
    segment.wave,
    segment.flux,
    [(4800, 4920)],
    ncols=1
)

To specify and plot several ranges, with labels, you can use

In [ ]:
sp.plot_spectrum_line_windows(
    segment.wave,
    segment.flux,
    [
        ("Hβ", 4830, 4890),
        ("Hα", 6540, 6585),
    ],
    valid_mask=segment.valid_mask,
    ncols=2,
)

Of course the ranges must be covered by the data, otherwise the plot will be empty.

To confirm the exact accepted formats for this function, you can run: 

In [ ]:
help(sp.plot_spectrum_line_windows)

## 4. Build a reviewed first-pass setup

Before running the PHOENIX fit, Spyctres can inspect the spectrum and suggest a sensible first-pass configuration.


In [ ]:
setup = sp.suggest_fit_setup(
    spec,
    mode="quicklook",
    intent="quicklook_classification",
)

print(setup.summary_text(include_hash=False))

This step is fast and does not load any PHOENIX templates. It shows which wavelength regions, masks, resolution assumptions, and parameter ranges Spyctres proposes to use.

Using a `FitSetup` is optional, but it is recommended for real data because it lets you review the assumptions before starting the fit.

## 5. Perform the fit using the PHOENIX templates

This is the slower part of this example. Before running it, check that Spyctres can find and use your PHOENIX library by running this command from the command line:

`spyctres doctor --require-phoenix`

`spyctres doctor` checks your Python environment, Spyctres installation, dependencies, and PHOENIX configuration. It only reports problems; it does not change anything.

When the spyctres command is not available on your shell path, you can instead run:

`python -m Spyctres.cli doctor --require-phoenix`

Set `RUN_FIT` to `True` when the setup check passes. Leave it as `False` when you only want to inspect the observed spectrum without loading PHOENIX templates.


In [ ]:
# Set this to True when PHOENIX is configured and you want to run the fit.
RUN_FIT = True

if RUN_FIT:
    # Fit the spectrum using the settings created previously.
    result = sp.fit_stellar_spectrum(
        spec,
        model="phoenix",
        setup=setup,
    )
    
    # Print the fitted parameters and the most important warnings.
    print(result.summary_text(include_hash=False, max_flags=5))
    
    # Compare the data and model in four useful diagnostic regions. 
    # Individual spectral features are easier to judge rather than looking
    # at one compressed plot of the entire spectrum.
    sp.plot_model_line_windows(
        result,
        windows=windows.selected[:4],
        segment=spec,
        title="Example 1: PHOENIX fit in diagnostic windows",
        ncols=2,
        figsize_per_panel=(7.2, 3.7),
    )
else:
    # Skip the fitting, just plot the windows
    sp.plot_spectrum_line_windows(
        spec.wave,
        spec.flux,
        windows.selected[:4],
        valid_mask=spec.valid_mask,
        title="Example 1: observed diagnostic line windows",
        ncols=2,
        figsize_per_panel=(7.2, 3.4),
    )
    
    print("RUN_FIT is False, so no PHOENIX templates were loaded. No fitting was performed.")


If the optional PHOENIX run reports a large reduced χ², read it as a quality flag rather than a single diagnosis. For high-S/N benchmark spectra the formal errors can be tiny, and remaining continuum/LSF/abundance/model-support differences can dominate χ² even when the broad classification is visually sensible.


## What to try next

Example 2 uses a more difficult X-SHOOTER spectrum and shows how to build explicit masks and fit local diagnostic lines.

For help on a public function:


In [ ]:
sp.describe_public_function("read_spectrum")
